In [2]:
import numpy as np

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score
)

import tensorflow as tf

from tensorflow.keras.models import Model

from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import EarlyStopping

from tensorflow.keras.regularizers import l2

In [3]:
X = np.load("../processed/X_no_norm.npy")
y = np.load("../processed/y_no_norm.npy")
subjects = np.load("../processed/subjects_no_norm.npy")

print(X.shape)
print(y.shape)

(4635, 7, 297)
(4635,)


In [4]:
unique_subjects = np.unique(subjects)

subject_kfold = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

In [5]:
def build_exact_model():

    inputs = Input(shape=(7,297))

    # D0 = 0.6
    x = Dropout(0.6)(inputs)

    # First LSTM
    x = LSTM(
        256,
        return_sequences=True,
        kernel_regularizer=l2(1e-4)
    )(x)

    # D1 = 0.2
    x = Dropout(0.2)(x)

    # Second LSTM
    x = LSTM(
        256,
        return_sequences=True,
        kernel_regularizer=l2(1e-4)
    )(x)

    # D2 = 0.1
    x = Dropout(0.1)(x)

    # Third LSTM
    x = LSTM(
        256,
        kernel_regularizer=l2(1e-4)
    )(x)

    # D3 = 0.2
    x = Dropout(0.2)(x)

    outputs = Dense(
        1,
        activation="sigmoid"
    )(x)

    model = Model(
        inputs,
        outputs
    )

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=0.001
    )

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [6]:
# Fold 1 only

train_sub_idx, test_sub_idx = next(
    subject_kfold.split(unique_subjects)
)

train_subjects = unique_subjects[
    train_sub_idx
]

test_subjects = unique_subjects[
    test_sub_idx
]

print("Train Subjects:", len(train_subjects))
print("Test Subjects :", len(test_subjects))

print(
    "Intersection:",
    np.intersect1d(
        train_subjects,
        test_subjects
    )
)

train_mask = np.isin(
    subjects,
    train_subjects
)

test_mask = np.isin(
    subjects,
    test_subjects
)

X_train = X[train_mask]
y_train = y[train_mask]

X_test = X[test_mask]
y_test = y[test_mask]

print("\nTrain Shape:")
print(X_train.shape)

print("\nTest Shape:")
print(X_test.shape)

print("\nTrain Labels:")
print(
    np.unique(
        y_train,
        return_counts=True
    )
)

print("\nTest Labels:")
print(
    np.unique(
        y_test,
        return_counts=True
    )
)

# ---------------------------------
# Standardization
# ---------------------------------

scaler = StandardScaler()

X_train_flat = X_train.reshape(-1, 297)
X_test_flat = X_test.reshape(-1, 297)

X_train_flat = scaler.fit_transform(
    X_train_flat
)

X_test_flat = scaler.transform(
    X_test_flat
)

X_train = X_train_flat.reshape(
    X_train.shape
)

X_test = X_test_flat.reshape(
    X_test.shape
)

# ---------------------------------
# Train Model
# ---------------------------------

model = build_exact_model()

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop]
)

# ---------------------------------
# Evaluation
# ---------------------------------

pred = model.predict(X_test)

pred = (pred > 0.5).astype(int)

from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        pred
    )
)

Train Subjects: 92
Test Subjects : 11
Intersection: []

Train Shape:
(4140, 7, 297)

Test Shape:
(495, 7, 297)

Train Labels:
(array([0, 1]), array([2083, 2057]))

Test Labels:
(array([0, 1]), array([254, 241]))
Epoch 1/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 22s 88ms/step - accuracy: 0.6498 - loss: 0.7083 - val_accuracy: 0.7174 - val_loss: 0.6334
Epoch 2/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 19s 74ms/step - accuracy: 0.7249 - loss: 0.6019 - val_accuracy: 0.7343 - val_loss: 0.5936
Epoch 3/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 11s 75ms/step - accuracy: 0.7544 - loss: 0.5535 - val_accuracy: 0.7899 - val_loss: 0.5079
Epoch 4/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 9s 76ms/step - accuracy: 0.7668 - loss: 0.5315 - val_accuracy: 0.7754 - val_loss: 0.5470
Epoch 5/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 71ms/step - accuracy: 0.7885 - loss: 0.5090 - val_accuracy: 0.7754 - val_loss: 0.5770
Epoch 6/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 11s 77ms/step - accuracy: 0.7893 - loss: 0.4961 - val_accuracy: 0.8188 - val_loss: 0.4930
Epoc